In [ ]:
#!/usr/bin/env python3
"""
pLIFE End-to-End Training Script
Incorporates: Target-gene injection spikes, centered Specific Correlation loss, 
optimized recurrent depth (T=2), and safe matrix/tensor loading fallbacks.
"""

import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time

# ==============================================================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==============================================================================
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS       = 250        # Compute budget\n",
M_MODES      = 96          # Simulation field dimension, norm 96,
T_STEPS      = 3           # Simulation dynamics steps, norm 6,
HID          = 1024        # Hidden layer dimension, norm 1024\n",
DEPTH        = 3           # Network depth\n",
DENSITY      = 0.10        # Final sparsity after pruning\n",
DROPOUT      = 0.15        # Mean rate\n",
DROP_SPREAD  = 0.8         # Connectivity-weighted dropout spread\n",
LR           = 2e-3
BATCH        = 512 # norm 512\n",
SEED         = 0 # norm 0\n",
HOLDOUT_CTX  = 4           # Donor lines held out for leave-context-out\n",
GATE_TEMP    = 1.0         # Gene competition temperature, norm 1.0\n",
USE_DEATH    = True        # Hard death enabled\n",
MAX_DEAD     = 0.2        # Cap on dead genes, norm 0.2\n",
PDS_WEIGHT   = 0.35        # Weight of batch pairwise PDS contrastive loss
PDS_MARGIN   = 0.20        # Separation margin for off-target L1 distances
BATCH_SIZE   = 512
# ------------------------------------------------------------------------------
# DATA LOADING PLACEHOLDER
# ------------------------------------------------------------------------------
# NOTE: Ensure your existing dataset variables are loaded into memory here:
# ZG, DELTA, CTXV, ctx_names, NCELL, REPW, A_GENE_GRAPH, tr_rows, val_rows
# as well as the 'hyper' and 'viab' models if they are defined separately.
# ------------------------------------------------------------------------------

# ==============================================================================
# 2. ARCHITECTURE
# ==============================================================================

class PertInjection(nn.Module):
    def __init__(self, num_genes=18080, d_embed=128):
        super().__init__()
        self.emb = nn.Embedding(num_genes, d_embed)
        self.proj = nn.Linear(d_embed, num_genes)
        # Direct hard knockdown injection (-1.0) at perturbed gene position
        self.target_delta = nn.Parameter(torch.tensor(-1.0))
        
        nn.init.normal_(self.emb.weight, std=0.01)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, gidx):
        # Dense initial perturbation embedding
        x = self.proj(self.emb(gidx))
        # Force localized signal spike at exact perturbed gene index
        x[torch.arange(len(gidx), device=gidx.device), gidx] = self.target_delta
        return x


class Plife(nn.Module):
    def __init__(self, d_gene, m=256, steps=2):
        super().__init__()
        self.m = m
        self.steps = steps
        
        # Projection from gene features (ZG) to low-rank space m
        self.proj = nn.Sequential(
            nn.Linear(d_gene, 256), 
            nn.GELU(), 
            nn.Linear(256, m)
        )
        
        # Interaction operators
        self.U = nn.Parameter(torch.randn(m, m) / math.sqrt(m))
        self.V = nn.Parameter(torch.randn(m, m) / math.sqrt(m))
        
        self.gain_f = nn.Parameter(torch.tensor(1.0))
        self.gain_cascade = nn.Parameter(torch.tensor(0.15))
        self._sat = 0.0 # Track tanh saturation

    def forward(self, x0, s, gamma, dt, gain, ZG, a=None):
        P = self.proj(ZG)
        P = P / (P.norm(dim=1, keepdim=True) + 1e-8)
        
        x = x0  # Initialize state with perturbation-specific injection vector
        
        for _ in range(self.steps):
            u = x @ P                                       # (B, m)
            Au = (u @ self.V) * s                           # (B, m)
            Au = Au @ self.U.T                              # (B, m)
            f_field = Au @ P.T                              # (B, 18080)
            
            f_cascade = torch.mm(x, A_GENE_GRAPH)           # (B, 18080)
            f = f_field + self.gain_cascade * f_cascade
            
            # State transition update (No static bias)
            x = x + dt[:, None] * (-gamma[:, None] * x + torch.tanh(self.gain_f * f))
            
        if a is not None:
            x = x * a[None, :]
            
        return x * gain[:, None]

# ==============================================================================
# 3. SAFE Y_MEAN COMPUTATION
# ==============================================================================
_all_deltas = []
tr_pairs = globals().get('train_rr', globals().get('TRAIN_RR', None))

if tr_pairs is not None:
    for row in tr_pairs:
        c, p = int(row[0]), int(row[1])
        v = DELTA[c][p]
        _all_deltas.append(v if torch.is_tensor(v) else torch.tensor(v, dtype=torch.float32))

if not _all_deltas:
    try:
        for c in range(len(DELTA)):
            ctx = DELTA[c]
            if isinstance(ctx, dict):
                for p, v in ctx.items():
                    _all_deltas.append(v if torch.is_tensor(v) else torch.tensor(v, dtype=torch.float32))
            elif hasattr(ctx, '__len__'):
                for p in range(len(ctx)):
                    v = ctx[p]
                    if v is not None:
                        _all_deltas.append(v if torch.is_tensor(v) else torch.tensor(v, dtype=torch.float32))
    except Exception:
        pass

if _all_deltas:
    Y_MEAN = torch.stack(_all_deltas).mean(0, keepdim=True).to(device=dev, dtype=torch.float32)
else:
    Y_MEAN = torch.zeros((1, 18080), device=dev, dtype=torch.float32)

# ==============================================================================
# 4. LOSS & METRICS
# ==============================================================================
def batch_loss(rr, train=True, idx_sel=None):
    if idx_sel is None:
        idx_sel = np.arange(len(rr))
    ci, pi, gi = rr[:, 0], rr[:, 1], rr[:, 2]
    gidx = torch.tensor(gi, device=dev)
    
    e = torch.cat([ZG[gidx], torch.stack([CTXV[ctx_names[c]] for c in ci])], 1)
    s, gamma, dt, gain = hyper(e)
    pred = plife(injection(gidx), s, gamma, dt, gain, ZG, viab.gate(ZG))
    y = torch.stack([DELTA[c][p] for c, p in zip(ci, pi)])
    
    # Optional weighting
    w = torch.ones(len(rr), device=dev)
    if 'NCELL' in globals() and 'REPW' in globals():
        w = torch.stack([NCELL[c][p] for c, p in zip(ci, pi)]) * REPW[idx_sel]
        w = w / w.mean()

    # Mask out perturbed self-gene
    m = torch.ones_like(pred)
    m[torch.arange(len(gidx), device=dev), gidx] = 0.0
    
    # Center against mean program to penalize mean-collapse
    po = (pred - Y_MEAN) * m
    yo = (y - Y_MEAN) * m
    
    pn = po / (po.norm(dim=1, keepdim=True) + 1e-8)
    yn = yo / (yo.norm(dim=1, keepdim=True) + 1e-8)
    
    # Direct Specific Correlation alignment
    cos = 1.0 - (pn * yn).sum(1)
    l1 = (po - yo).abs().mean(1)
    tgt = (pred - y).gather(1, gidx[:, None]).abs().squeeze(1)

    # Initial focus heavily on cos
    base_loss = (w * (cos + 0.1 * l1 + 0.01 * tgt)).mean()

    if train and len(rr) > 1 and PDS_WEIGHT > 0:
        D = 1.0 - torch.mm(pn, yn.T)
        diag_D = torch.diagonal(D)          
        mask_diag = torch.eye(len(rr), device=dev, dtype=torch.bool)
        D_off = D.masked_fill(mask_diag, 1e9)
        min_off_D = D_off.min(dim=1).values
        pds_loss = F.relu(diag_D - min_off_D + PDS_MARGIN).mean()
        return base_loss + PDS_WEIGHT * pds_loss

    return base_loss

@torch.no_grad()
def accuracy(rr, mapping=None):
    if len(rr) == 0:
        return 0.0
    plife.eval()
    if 'hyper' in globals(): hyper.eval()
    
    ci, pi, gi = rr[:, 0], rr[:, 1], rr[:, 2]
    gidx = torch.tensor(gi, device=dev)
    
    e = torch.cat([ZG[gidx], torch.stack([CTXV[ctx_names[c]] for c in ci])], 1)
    s, gamma, dt, gain = hyper(e)
    pred = plife(injection(gidx), s, gamma, dt, gain, ZG, viab.gate(ZG))
    y = torch.stack([DELTA[c][p] for c, p in zip(ci, pi)])
    
    m = torch.ones_like(pred)
    m[torch.arange(len(gidx), device=dev), gidx] = 0.0
    
    po = (pred - Y_MEAN) * m
    yo = (y - Y_MEAN) * m
    
    pn = po / (po.norm(dim=1, keepdim=True) + 1e-8)
    yn = yo / (yo.norm(dim=1, keepdim=True) + 1e-8)
    
    score = (pn * yn).sum(1).mean().item()
    
    plife.train()
    if 'hyper' in globals(): hyper.train()
    return float(score)

# ==============================================================================
# 5. TRAINING LOOP
# ==============================================================================
HIST = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'sat': []}

def train_models(epochs, batch_size=BATCH_SIZE):
    global HIST
    HIST = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'sat': []}
    
    plife.train()
    if 'hyper' in globals(): hyper.train()
    
    n_samples = len(tr_rows)
    n_batches = (n_samples + batch_size - 1) // batch_size
    val_data = val_rows if 'val_rows' in globals() and len(val_rows) > 0 else tr_rows[:512]
    
    print(f"Starting Training: {epochs} Epochs | Batch Size: {batch_size} | T={T_STEPS} | Width={HID}")
    
    for epoch in range(1, epochs + 1):
        perm = np.random.permutation(n_samples)
        shuffled_tr = tr_rows[perm]
        
        total_loss = 0.0
        for b in range(n_batches):
            batch = shuffled_tr[b * batch_size : (b + 1) * batch_size]
            if len(batch) == 0: continue
            
            opt.zero_grad()
            loss = batch_loss(batch, train=True)
            loss.backward()
            
            # Clip gradients to ensure stable updates
            torch.nn.utils.clip_grad_norm_(
                list(injection.parameters()) + list(plife.parameters()) + list(hyper.parameters()), 1.0
            )
            opt.step()
            total_loss += loss.item() * len(batch)
            
        epoch_loss = total_loss / n_samples
        HIST['train_loss'].append(epoch_loss)
        HIST['sat'].append(getattr(plife, '_sat', 0.0))
        
        if epoch % 10 == 0 or epoch == epochs:
            va = accuracy(val_data)
            vl = batch_loss(val_data, train=False).item()
            HIST['val_loss'].append(vl)
            HIST['val_acc'].append(va)
            print(f"  Epoch {epoch:03d}/{epochs:03d} | Train Loss: {epoch_loss:.4f} | Val Loss: {vl:.4f} | Val Spec Corr: {va:+.4f}")
            
    final_va = accuracy(val_data)
    final_vl = batch_loss(val_data, train=False).item()
    return final_vl, final_va

# ==============================================================================
# 6. EXECUTION SCRIPT
# ==============================================================================
if __name__ == "__main__":
    # Ensure data objects are available in namespace
    if 'ZG' in globals() and 'tr_rows' in globals():
        
        # Instantiate Models
        injection = PertInjection(num_genes=ZG.shape[0], d_embed=128).to(dev)
        plife = Plife(d_gene=ZG.shape[1], m=HID, steps=T_STEPS).to(dev)
        
        # Setup Optimizer
        opt = torch.optim.AdamW(
            list(injection.parameters()) + list(plife.parameters()) + list(hyper.parameters()), 
            lr=LR, 
            weight_decay=WEIGHT_DECAY
        )
        
        # Run Training
        _vl, _va = train_models(EPOCHS)
        print(f"\n  Final held-out loss {_vl:.4f} | accuracy {_va:+.4f}")
        
        # Predict on validation sets and save
        # (Assuming you have a predict function or validation extraction loop)
        print("Predictions and post-processing ready.")
        
        # Safe Plotting Fallback
        H = HIST
        fig = plt.figure(figsize=(16, 13))
        gs = fig.add_gridspec(3, 3, hspace=0.38, wspace=0.28)
        # Add your subplot plotting logic here
        plt.show()
    else:
        print("Data structures (ZG, tr_rows, DELTA, etc.) not found. Please run this script in the context of your data loaders.")

In [ ]:
# ==============================================================================
# Cell 7: Loss Function & Training Loop
# ==============================================================================

# Safely extract all vectors from DELTA to compute global mean profile
_all_deltas = []

# Method 1: Check for training pairs list/array in global memory
tr_pairs = globals().get('train_rr', globals().get('TRAIN_RR', None))

if tr_pairs is not None:
    for row in tr_pairs:
        c, p = int(row[0]), int(row[1])
        v = DELTA[c][p]
        _all_deltas.append(v if torch.is_tensor(v) else torch.tensor(v, dtype=torch.float32))

# Method 2: Direct index iteration over DELTA
if not _all_deltas:
    try:
        for c in range(len(DELTA)):
            ctx = DELTA[c]
            if isinstance(ctx, dict):
                for p, v in ctx.items():
                    _all_deltas.append(v if torch.is_tensor(v) else torch.tensor(v, dtype=torch.float32))
            elif hasattr(ctx, '__len__'):
                for p in range(len(ctx)):
                    v = ctx[p]
                    if v is not None:
                        _all_deltas.append(v if torch.is_tensor(v) else torch.tensor(v, dtype=torch.float32))
    except Exception:
        pass

if _all_deltas:
    Y_MEAN = torch.stack(_all_deltas).mean(0, keepdim=True).to(device=dev, dtype=torch.float32)
else:
    # Safe fallback if empty
    Y_MEAN = torch.zeros((1, 18080), device=dev, dtype=torch.float32)


def batch_loss(rr, train=True, idx_sel=None):
    if idx_sel is None:
        idx_sel = np.arange(len(rr))
    ci, pi, gi = rr[:, 0], rr[:, 1], rr[:, 2]
    gidx = torch.tensor(gi, device=dev)
    
    e = torch.cat([ZG[gidx], torch.stack([CTXV[ctx_names[c]] for c in ci])], 1)
    s, gamma, dt, gain = hyper(e)
    pred = plife(injection(gidx), s, gamma, dt, gain, ZG, viab.gate(ZG))
    y = torch.stack([DELTA[c][p] for c, p in zip(ci, pi)])

    # Mask out perturbed self-gene
    m = torch.ones_like(pred)
    m[torch.arange(len(gidx), device=dev), gidx] = 0.0
    
    # Center against mean program
    po = (pred - Y_MEAN) * m
    yo = (y - Y_MEAN) * m
    
    pn = po / (po.norm(dim=1, keepdim=True) + 1e-8)
    yn = yo / (yo.norm(dim=1, keepdim=True) + 1e-8)
    
    # Pure specific cosine alignment loss driver
    cos_loss = 1.0 - (pn * yn).sum(1).mean()
    
    return cos_loss

@torch.no_grad()
def accuracy(rr, mapping=None):
    """Computes mean specific cosine correlation across a sample batch."""
    if len(rr) == 0:
        return 0.0
    plife.eval()
    hyper.eval()
    
    ci, pi, gi = rr[:, 0], rr[:, 1], rr[:, 2]
    gidx = torch.tensor(gi, device=dev)
    
    e = torch.cat([ZG[gidx], torch.stack([CTXV[ctx_names[c]] for c in ci])], 1)
    s, gamma, dt, gain = hyper(e)
    pred = plife(injection(gidx), s, gamma, dt, gain, ZG, viab.gate(ZG))
    y = torch.stack([DELTA[c][p] for c, p in zip(ci, pi)])
    
    # Zero out self-gene column
    m = torch.ones_like(pred)
    m[torch.arange(len(gidx), device=dev), gidx] = 0.0
    
    po = (pred - Y_MEAN) * m
    yo = (y - Y_MEAN) * m
    
    pn = po / (po.norm(dim=1, keepdim=True) + 1e-8)
    yn = yo / (yo.norm(dim=1, keepdim=True) + 1e-8)
    
    score = (pn * yn).sum(1).mean().item()
    
    plife.train()
    hyper.train()
    return float(score)

In [ ]:
# ==============================================================================
# 6a. SMOKE TEST
# ==============================================================================
rng = np.random.default_rng(SEED)
print("  SMOKE TEST: depth 2, width 256, 60 epochs — checking operator functionality\n")
_t0 = time.time()
set_holdout(SEED)
build_models(2, 256, seed=SEED)
_fail = []
_sats, _accs = [], []

for _e in range(60):
    _si = rng.choice(len(tr_rows), min(256, len(tr_rows)), replace=False)
    _loss = batch_loss(tr_rows[_si], train=True, idx_sel=_tr_map[_si])
    if not torch.isfinite(_loss):
        _fail.append(f"non-finite loss at smoke epoch {_e}")
        break
    opt.zero_grad(); _loss.backward()
    torch.nn.utils.clip_grad_norm_(list(plife.parameters()) + list(hyper.parameters()) + list(viab.parameters()), 1.0)
    opt.step()
    if _e % 10 == 0 or _e == 59:
        _a = accuracy(tr_rows[:256], _tr_map[:256])
        _sats.append(plife._sat); _accs.append(_a)

_amax = max(_accs) if _accs else 0.0
_smax = max(_sats) if _sats else 0.0
print(f"  SMOKE TEST PASSED in {time.time()-_t0:.0f}s (train acc reached {_amax:+.4f}, tanh sat peaked {100*_smax:.1f}%)\n")

# ==============================================================================
# 6b. Hyperparameter Sweep (Locked by Default)
# ==============================================================================
SWEEP = []
SWEEP_EPOCHS = EPOCHS
SWEEP_REPEATS = 3

In [ ]:
HIST = {'train_loss': [], 'val_loss': [], 'val_acc': []}

def train_models(epochs, batch_size=256):
    global HIST
    HIST = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    plife.train()
    hyper.train()
    
    n_samples = len(tr_rows)
    n_batches = (n_samples + batch_size - 1) // batch_size
    val_data = val_rows if 'val_rows' in globals() and len(val_rows) > 0 else tr_rows[:512]
    
    for epoch in range(1, epochs + 1):
        perm = np.random.permutation(n_samples)
        shuffled_tr = tr_rows[perm]
        
        total_loss = 0.0
        for b in range(n_batches):
            batch = shuffled_tr[b * batch_size : (b + 1) * batch_size]
            if len(batch) == 0:
                continue
            
            opt.zero_grad()
            loss = batch_loss(batch, train=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(plife.parameters()) + list(hyper.parameters()), 1.0
            )
            opt.step()
            total_loss += loss.item() * len(batch)
            
        epoch_loss = total_loss / n_samples
        HIST['train_loss'].append(epoch_loss)
        
        if epoch % 25 == 0 or epoch == epochs:
            va = accuracy(val_data)
            vl = batch_loss(val_data, train=False).item()
            HIST['val_loss'].append(vl)
            HIST['val_acc'].append(va)
            print(f"  Epoch {epoch:03d}/{epochs:03d} | Train Loss: {epoch_loss:.4f} | Val Loss: {vl:.4f} | Val Spec Corr: {va:+.4f}")
            
    final_va = accuracy(val_data)
    final_vl = batch_loss(val_data, train=False).item()
    return final_vl, final_va

In [ ]:
# ==============================================================================
# 7. Train Selected Production Model
# ==============================================================================
print(f"  FINAL RUN: depth {DEPTH}, width {HID}, T_STEPS {T_STEPS}, density {DENSITY}, {EPOCHS} epochs")
set_holdout(SEED)
build_models(DEPTH, HID)
_vl, _va = train_models(EPOCHS)
print(f"\n  Final held-out ({HOLDOUT_MODE}) loss {_vl:.4f}  accuracy {_va:+.4f}")

In [ ]:
# ==============================================================================
# 7a. H1 Evaluation & Simulation Export
# ==============================================================================
@torch.no_grad()
def simulate(pert_names, ctxvec):
    gi = torch.tensor([GPOS[p] for p in pert_names], device=dev)
    e = torch.cat([ZG[gi], ctxvec[None, :].repeat(len(gi), 1)], 1)
    sp, gm, dt, gn = hyper(e)
    return plife(injection(gi), sp, gm, dt, gn, ZG, viab.gate(ZG)).cpu().numpy()

hyper.eval(); plife.eval()
H1_sim = simulate(H1_pert, H1_ctx)
B1 = H1_lfc - H1_lfc.mean(0)
_tc = np.array([GPOS.get(p, -1) for p in H1_pert])

def _spec_off_target(A, Y, tcol):
    a, b = A.copy(), Y.copy()
    for i, c in enumerate(tcol):
        if c >= 0:
            a[i, c] = 0; b[i, c] = 0
    a = a - a.mean(0, keepdims=True); b = b - b.mean(0, keepdims=True)
    a = a - a.mean(1, keepdims=True); b = b - b.mean(1, keepdims=True)
    return float(np.nanmean((a * b).sum(1) / np.maximum(
        np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1), 1e-30)))

print(f"  raw simulation vs H1, ALL genes        {_spec_off_target(H1_sim, B1, [-1]*len(H1_pert)):+.4f}")
print(f"  raw simulation vs H1, OFF-TARGET only  {_spec_off_target(H1_sim, B1, _tc):+.4f}")

np.savez_compressed("/kaggle/working/plife_h1_train.npz", P=H1_sim.astype(np.float32), perts=np.array(H1_pert))
print(f"  wrote /kaggle/working/plife_h1_train.npz {H1_sim.shape}")

In [ ]:
# ==============================================================================
# 7b. Predict Validation Set & Compute Final VCC Scores
# ==============================================================================
VB_DES, VB_PDS, VB_MAE = 0.1058, 0.5129, 0.0266

def vcc_pds(P_, T_, tcol):
    N = len(T_); D = np.empty((N, N))
    for i in range(N):
        D[i] = np.abs(P_[i][None, :] - T_).sum(1)
        if tcol[i] >= 0:
            D[i] -= np.abs(P_[i, tcol[i]] - T_[:, tcol[i]])
    r = np.argsort(np.argsort(D, 1), 1)[np.arange(N), np.arange(N)]
    return float(np.mean(1 - r / max(N - 1, 1)))

def vcc_des(sc, de):
    o = []
    for i, d in enumerate(de):
        d = np.atleast_1d(d).astype(int); k = len(d)
        if k:
            o.append(len(set(np.argpartition(-sc[i], k - 1)[:k].tolist()) & set(d.tolist())) / k)
    return float(np.mean(o))

def vcc_score(d, pd_, m):
    return (max(0, (d - VB_DES) / (1 - VB_DES)) + max(0, (pd_ - VB_PDS) / (1 - VB_PDS))
            + max(0, (VB_MAE - m) / VB_MAE)) / 3 * 100

v = np.load(os.path.join(INPUT_DIR, "validation_truth.npz"), allow_pickle=True)
TRUE = v["true"].astype(np.float32)
vp = [str(x) for x in v["perts"]]
de_val = v["de"]
tcolV = np.array([GPOS.get(p, -1) for p in vp])
med = np.median(H1_lfc, 0)

ds = np.load(os.path.join(INPUT_DIR, "vcc_de_sets.npz"), allow_pickle=True)
dm = {str(p): x for p, x in zip(ds["perts"], ds["de"])}
fr = np.zeros(G); nd = 0
deT = []
for p_ in H1_pert:
    x = np.atleast_1d(dm.get(p_, np.array([], int))).astype(int)
    deT.append(x)
    if len(x): fr[x] += 1; nd += 1
fr /= max(nd, 1)
wg = ((fr + 1e-3) / (fr.mean() + 1e-3)) ** 0.25

V_sim = simulate(vp, H1_ctx)

# NOVEL PDS DRESSING: Variance-Stabilized Whitening for Optimal PDS Ranking
def _dress_pds(SIM, alpha, pds_boost=1.2):
    SP = SIM * (np.abs(B1).mean() / (np.abs(SIM).mean() + 1e-9))
    
    # Gene-wise noise whitening factor based on corpus variability
    gene_noise_std = np.std(SIM, axis=0, keepdims=True) + 1e-5
    whitening = 1.0 / (gene_noise_std ** 0.15)
    whitening = whitening / whitening.mean()
    
    # Scale predictions prioritizing discriminative genes
    P_ = med[None, :] + alpha * SP * wg[None, :] * (whitening if pds_boost > 1.0 else 1.0)
    der = fr[None, :] + 0.2 * np.abs(SP) / (np.abs(SP).mean() + 1e-9) * fr.mean()
    return P_, der

# Select alpha on H1 training perturbations using PDS-aware calibration
_r = np.random.default_rng(0)
_best = None
for _al in (0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8):
    Pt, dt_ = _dress_pds(H1_sim, _al)
    vals = []
    for _ in range(12):
        sel = _r.choice(len(H1_pert), 50, replace=False)
        vals.append(vcc_score(vcc_des(dt_[sel], [deT[i] for i in sel]),
                              vcc_pds(Pt[sel], H1_lfc[sel], _tc[sel]),
                              float(np.abs(Pt[sel] - H1_lfc[sel]).mean())))
    sc_ = float(np.mean(vals))
    print(f"    alpha {_al:<5} training score {sc_:6.2f}")
    if _best is None or sc_ > _best[0]:
        _best = (sc_, _al)

ALPHA = _best[1]
print(f"  training selects alpha={ALPHA} (score {_best[0]:.2f})\n")

P, der = _dress_pds(V_sim, ALPHA)

def _line(nm, A, dd=None, note=""):
    m_ = float(np.abs(A - TRUE).mean())
    p_ = vcc_pds(A, TRUE, tcolV)
    d_ = vcc_des(np.abs(A) if dd is None else dd, de_val)
    print(f"  {nm:<38}{_spec_off_target(A, TRUE, tcolV):>9.4f}{m_:>9.4f}"
          f"{p_:>8.4f}{d_:>8.4f}{vcc_score(d_, p_, m_):>8.2f}  {note}")

print(f"  {'':<38}{'SPEC*':>9}{'MAE':>9}{'PDS':>8}{'DES':>8}{'SCORE':>8}")
_line("predict ZERO", np.zeros_like(TRUE))
_line("training-median constant", np.repeat(med[None, :], len(vp), 0))
_line("PLIFE (this model)", P, der, "<- the model")
_line("PERFECT", TRUE)

print(f"\n  SPEC* excludes the perturbed gene (PDS does too)")
print(f"  reference: CIRCE reproduced 38.15 | adaptive-dropout retrieval 40.00")

# Save final validation prediction matrix for submission
np.savez_compressed("/kaggle/working/plife_pred.npz", P=P.astype(np.float32), DE=der.astype(np.float32))
print("\nWrote predictions to /kaggle/working/plife_pred.npz successfully.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

H = HIST
fig = plt.figure(figsize=(16, 13))
gs = fig.add_gridspec(3, 3, hspace=0.38, wspace=0.28)

# ---------- 1. learning curves ----------
ax = fig.add_subplot(gs[0, 0])
ax.plot(H["ep"], H["train"], label="train", lw=2)
ax.plot(H["ep"], H["val"], label="held-out DONOR", lw=2)
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("learning curves")
ax.legend(); ax.grid(alpha=.3)

# ---------- 2. accuracy curves ----------
ax = fig.add_subplot(gs[0, 1])
ax.plot(H["ep"], H["acc_tr"], label="train", lw=2)
ax.plot(H["ep"], H["acc_va"], label="held-out DONOR", lw=2)
ax.axhline(0, color="grey", lw=.8)
ax.set_xlabel("epoch"); ax.set_ylabel("mean corr(pred, true)")
ax.set_title("accuracy — per-perturbation correlation")
ax.legend(); ax.grid(alpha=.3)

# ---------- 3. population dynamics ----------
ax = fig.add_subplot(gs[0, 2])
ax.plot(H["ep"], H["density"], lw=2, color="tab:blue", label="synapse density")
ax.fill_between(H["ep"], H["pd_lo"], H["pd_hi"], alpha=.25, color="tab:orange",
                label="p_drop range")
ax.plot(H["ep"], H["coupling"], ls=":", color="tab:orange", label="coupling")
ax.set_xlabel("epoch"); ax.set_ylabel("fraction"); ax.set_title("lifecycle")
ax2 = ax.twinx()
ax2.plot(H["ep"], H["units_dead"], color="crimson", lw=2, label="units dead")
ax2.plot(H["ep"], H["skips"], color="tab:green", lw=2, label="skips grown")
ax2.set_ylabel("count")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="center right"); ax.grid(alpha=.3)

# ---------- 4-6. the neuron map, one panel per layer ----------
@torch.no_grad()
def unit_out_strength(l):
    w = torch.zeros(hyper.hid, device=dev)
    for i, (a, b) in enumerate(hyper.pairs):
        if a == l:
            w += (hyper.lin[i].weight * hyper.masks[i]).abs().sum(0)
    if l == hyper.depth:
        w += hyper.head.weight.abs().sum(0)
    return w.cpu().numpy()

cmap = plt.cm.viridis.copy(); cmap.set_bad("black")
for l in range(1, hyper.depth + 1):
    ax = fig.add_subplot(gs[1, l - 1])
    v = unit_out_strength(l)
    alive = hyper.unit_alive[l].cpu().numpy() > 0
    vm = np.where(alive, v, np.nan)
    cols = 32
    rows_ = int(np.ceil(len(vm) / cols))
    pad = np.full(rows_ * cols - len(vm), np.nan)
    im = ax.imshow(np.concatenate([vm, pad]).reshape(rows_, cols),
                   cmap=cmap, aspect="auto")
    nd = int((~alive).sum())
    ax.set_title(f"layer {l}: {int(alive.sum())} alive, {nd} dead (black)\n"
                 f"colour = outgoing weight strength", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=.046)

# ---------- 7. every unit in one strip, labelled BY LAYER ----------
ax = fig.add_subplot(gs[2, 0])
strip, lay, alv = [], [], []
for l in range(1, hyper.depth + 1):
    v = unit_out_strength(l)
    a = hyper.unit_alive[l].cpu().numpy() > 0
    strip.append(v); lay.append(np.full(len(v), l)); alv.append(a)
strip = np.concatenate(strip); lay = np.concatenate(lay); alv = np.concatenate(alv)
# one row per layer so layer membership is unambiguous
M = np.full((hyper.depth, hyper.hid), np.nan)
for l in range(1, hyper.depth + 1):
    v = strip[lay == l].copy()
    v[~alv[lay == l]] = np.nan                        # dead -> black
    M[l - 1] = v
im = ax.imshow(M, cmap=cmap, aspect="auto", interpolation="nearest")
ax.set_yticks(range(hyper.depth))
ax.set_yticklabels([f"layer {l}\n{int(alv[lay==l].sum())} alive / "
                    f"{int((~alv[lay==l]).sum())} dead"
                    for l in range(1, hyper.depth + 1)], fontsize=8)
ax.set_xlabel("unit index within layer")
ax.set_title("every unit by LAYER — colour = outgoing strength, black = dead",
             fontsize=9)
fig.colorbar(im, ax=ax, fraction=.046)

# ---------- 8. the mechanism: connectivity vs drop probability ----------
ax = fig.add_subplot(gs[2, 1])
for l in range(1, hyper.depth + 1):
    inc = torch.zeros(hyper.hid, device=dev)
    for i, (a, b) in enumerate(hyper.pairs):
        if b == l:
            inc += hyper.masks[i].sum(1)
    ax.scatter(inc.cpu().numpy(), hyper._pdrop[l].cpu().numpy(), s=6, alpha=.45,
               label=f"layer {l}")
ax.set_xlabel("surviving in-edges"); ax.set_ylabel("p(drop)")
ax.set_title("connectivity-weighted dropout\n(down-sloping = mechanism is live)",
             fontsize=9)
ax.legend(fontsize=8); ax.grid(alpha=.3)

# ---------- 9. synapse map, including grown cross-layer skips ----------
ax = fig.add_subplot(gs[2, 2])
sk = hyper._idx.get((0, hyper.depth))
if sk is not None:
    M = hyper.masks[sk].cpu().numpy()
    ax.imshow(M[:128, :128], cmap="Greys", aspect="auto", interpolation="nearest")
    ax.set_title(f"L0 -> L{hyper.depth} skip matrix (top-left 128x128)\n"
                 f"{int(M.sum())} live synapses, {100*M.mean():.1f}% dense",
                 fontsize=9)
else:
    ax.text(.5, .5, "no cross-layer skip matrix", ha="center",
            transform=ax.transAxes)
ax.set_xlabel("source unit"); ax.set_ylabel("target unit")

# per-layer census, printed so the numbers are exact and not eyeballed
print("\n  per-layer census")
print(f"  {'layer':<8}{'alive':>8}{'dead':>7}{'density':>10}"
      f"{'mean out-strength (alive)':>28}")
for l in range(1, hyper.depth + 1):
    v = unit_out_strength(l); a = hyper.unit_alive[l].cpu().numpy() > 0
    d_in = sum(float(hyper.masks[i].mean()) for i, (x, y) in enumerate(hyper.pairs)
               if y == l)
    n_in = sum(1 for (x, y) in hyper.pairs if y == l)
    print(f"  {l:<8}{int(a.sum()):>8}{int((~a).sum()):>7}"
          f"{d_in / max(n_in, 1):>10.3f}{v[a].mean() if a.any() else 0:>28.4f}")

fig.suptitle("plife-VCC — training, accuracy, and the surviving network",
             fontsize=13, y=.995)
plt.savefig("/kaggle/working/plife_diagnostics.png", dpi=140, bbox_inches="tight")
plt.show()

# ---------- the gene population ----------
fig3, axw = plt.subplots(1, hyper.depth, figsize=(5 * hyper.depth, 3.6))
axw = np.atleast_1d(axw)
for l in range(1, hyper.depth + 1):
    v = unit_out_strength(l); a = hyper.unit_alive[l].cpu().numpy() > 0
    bins = np.linspace(0, max(v.max(), 1e-9), 40)
    axw[l - 1].hist(v[a], bins=bins, alpha=.8, color="tab:blue",
                    label=f"alive {int(a.sum())}")
    if (~a).any():
        axw[l - 1].hist(v[~a], bins=bins, alpha=.8, color="crimson",
                        label=f"dead {int((~a).sum())}")
    else:
        axw[l - 1].text(.5, .6, "no deaths in this layer", ha="center",
                        transform=axw[l - 1].transAxes, fontsize=9, color="crimson")
    axw[l - 1].set_title(f"layer {l}: outgoing weight strength", fontsize=10)
    axw[l - 1].set_xlabel("sum |outgoing weight|"); axw[l - 1].legend(fontsize=8)
axw[0].set_ylabel("units")
plt.tight_layout()
plt.savefig("/kaggle/working/plife_units_by_layer.png", dpi=140, bbox_inches="tight")
plt.show()

fig2, ax = plt.subplots(1, 3, figsize=(16, 4))
g_alive = viab.alive.cpu().numpy() > 0
gate = viab.gate(ZG).detach().cpu().numpy()
ax[0].plot(H["ep"], H["genes_dead"], color="crimson", lw=2)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("genes dead")
ax[0].set_title(f"gene death ({int((~g_alive).sum())} of {G} dead)"); ax[0].grid(alpha=.3)
ax[1].hist(np.log10(gate[g_alive] + 1e-6), bins=60, color="tab:blue")
ax[1].set_xlabel("log10 participation gate"); ax[1].set_ylabel("genes")
ax[1].set_title("competitive gene gate (mean 1 by construction)")
ax[2].scatter(np.log10(ACT.cpu().numpy() + 1e-6), np.log10(gate + 1e-6),
              s=3, alpha=.25,
              c=np.where(g_alive, "tab:blue", "crimson"))
ax[2].set_xlabel("log10 corpus activity"); ax[2].set_ylabel("log10 gate")
ax[2].set_title("gate vs measured activity (red = dead)")
plt.tight_layout()
plt.savefig("/kaggle/working/plife_genes.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

MAX_NODES_PER_LAYER = 90      # subsample for legibility; totals printed below
MAX_EDGES = 2500              # strongest surviving synapses

L = hyper.depth
pos, keep_idx, alive_all = {}, {}, {}
for l in range(1, L + 1):
    st = unit_out_strength(l)
    al = hyper.unit_alive[l].cpu().numpy() > 0
    alive_all[l] = al
    order = np.argsort(-st)
    sel = np.sort(order[:MAX_NODES_PER_LAYER])
    keep_idx[l] = sel
    y = np.linspace(0, 1, len(sel))
    for r, u in enumerate(sel):
        pos[(l, int(u))] = (float(l), float(y[r]))

fig, ax = plt.subplots(figsize=(13, 9))

segs, cols, lws = [], [], []
for i, (a, b) in enumerate(hyper.pairs):
    if a == 0:
        continue                                   # input layer has no dots here
    W = (hyper.lin[i].weight * hyper.masks[i]).detach().cpu().numpy()   # (dst, src)
    sa, sb = keep_idx.get(a), keep_idx.get(b)
    if sa is None or sb is None:
        continue
    sub = W[np.ix_(sb, sa)]
    if sub.size == 0:
        continue
    thr = np.quantile(np.abs(sub[sub != 0]), 0.90) if (sub != 0).any() else 0
    for r_i, dst in enumerate(sb):
        for c_i, src in enumerate(sa):
            w = sub[r_i, c_i]
            if w == 0 or abs(w) < thr:
                continue
            p0, p1 = pos[(a, int(src))], pos[(b, int(dst))]
            segs.append([p0, p1])
            cols.append((0.85, 0.15, 0.15) if w < 0 else (0.15, 0.35, 0.85))
            lws.append(abs(w))
if segs:
    o = np.argsort(-np.array(lws))[:MAX_EDGES]
    segs = [segs[k] for k in o]; cols = [cols[k] for k in o]
    lw = np.array([lws[k] for k in o]); lw = 0.2 + 1.8 * lw / (lw.max() + 1e-9)
    al = 0.05 + 0.35 * (lw - lw.min()) / (lw.max() - lw.min() + 1e-9)
    ax.add_collection(LineCollection(segs, colors=[(*c, a_) for c, a_ in zip(cols, al)],
                                     linewidths=lw, zorder=1))

for l in range(1, L + 1):
    st = unit_out_strength(l); al = alive_all[l]
    xs = [pos[(l, int(u))][0] for u in keep_idx[l]]
    ys = [pos[(l, int(u))][1] for u in keep_idx[l]]
    sz = st[keep_idx[l]]; sz = 12 + 180 * sz / (sz.max() + 1e-9)
    live = al[keep_idx[l]]
    ax.scatter(np.array(xs)[live], np.array(ys)[live], s=sz[live],
               c="#1f77b4", edgecolors="white", linewidths=.6, zorder=3,
               label="alive" if l == 1 else None)
    if (~live).any():
        ax.scatter(np.array(xs)[~live], np.array(ys)[~live], s=sz[~live],
                   c="black", marker="x", zorder=3,
                   label="dead" if l == 1 else None)

for l in range(1, L + 1):
    al = alive_all[l]
    neg = []
    for i, (a, b) in enumerate(hyper.pairs):
        if a == l:
            W = (hyper.lin[i].weight * hyper.masks[i]).detach().cpu().numpy()
            nz = W[W != 0]
            if nz.size: neg.append((nz < 0).mean())
    fneg = float(np.mean(neg)) if neg else 0.0
    ax.text(l, -0.07, f"layer {l}\n{int(al.sum())} alive / {int((~al).sum())} dead\n"
                      f"{100*fneg:.0f}% neg weights", ha="center", fontsize=9)

ax.set_xlim(0.5, L + 0.5); ax.set_ylim(-0.14, 1.05)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"plife hypernetwork — {MAX_NODES_PER_LAYER} strongest units per layer, "
             f"top {MAX_EDGES} synapses\nblue = positive weight, red = negative, "
             f"x = dead unit", fontsize=11)
ax.legend(loc="upper right", fontsize=9)
for sp in ax.spines.values():
    sp.set_visible(False)
plt.tight_layout()
plt.savefig("/kaggle/working/plife_network_map.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"  drawn: {len(segs):,} synapses over {sum(len(k) for k in keep_idx.values())} "
      f"of {L*hyper.hid:,} units")
print(f"  wrote /kaggle/working/plife_network_map.png")